In [1]:
import datasetsforecast
from datasetsforecast.m4 import M4
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS, NHITS

In [2]:
group = 'Daily'
await M4.async_download('data', group=group)
df, *_ = M4.load(directory='data', group=group)

In [3]:
df['ds'] = df.ds.astype(int)

In [4]:
# get the train indexes
train_indexes = (
    df.groupby('unique_id')
    .size()
    .reset_index(name='count')
    .assign(max_index=lambda x:(x['count']*0.8).astype(int))
)

# Merge the train_indexes with the original dataframe
df_with_max_index = df.merge(train_indexes[['unique_id', 'max_index']], on='unique_id', how='left')

# Create the mask based on the condition
train_mask = df_with_max_index.apply(lambda row: row['ds'] <= row['max_index'], axis=1)

# Apply the mask to create train and test dataframes
Y_train_df = df[train_mask]
Y_test_df = df[~train_mask]

print(f"Training set shape: {Y_train_df.shape}")
print(f"Test set shape: {Y_test_df.shape}")


Training set shape: (8017108, 3)
Test set shape: (2006728, 3)


In [5]:
Y_train_df.groupby('unique_id').size().max()

7946

In [6]:
horizon = 500
models = [NBEATS(input_size=2 * horizon, h=horizon, max_steps=10)]
nf = NeuralForecast(models=models, freq=1)

INFO:lightning_fabric.utilities.seed:Seed set to 1


In [7]:
nf.fit(df=Y_train_df)

/Users/scott.mckean/Library/Application Support/hatch/env/virtual/mmf-sa/o6PZDoNu/full/lib/python3.10/site-packages/neuralforecast/common/_base_model.py:208: UserWarning: val_check_steps is greater than max_steps, setting val_check_steps to max_steps.
  warnings.warn(
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (mps), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.callbacks.model_summary:
  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 7.2 M  | train
-------------------------------------------------------
5.7 M     Trainable params
1.5 M     Non-trainable params
7.2 M     Total par

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=10` reached.


In [12]:
Y_hat_df = nf.predict().reset_index()

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (mps), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


Predicting: |          | 0/? [00:00<?, ?it/s]

/Users/scott.mckean/Library/Application Support/hatch/env/virtual/mmf-sa/o6PZDoNu/full/lib/python3.10/site-packages/neuralforecast/core.py:196: FutureWarning:

In a future version the predictions will have the id as a column. You can set the `NIXTLA_ID_AS_COL` environment variable to adopt the new behavior and to suppress this warning.



In [13]:
# Fonts: #FF3621, #1B3139 
# Background: #F9F7F4
# Basic Palette: #98102A, #FFAB00, #00A972, #2272B4, #303F47, 

In [14]:
import plotly.io as pio
import plotly.graph_objects as go

# Define the Databricks-inspired theme
databricks_theme = {
    'layout': {
        'font': {
            'family': 'DM Sans, sans-serif',
            'color': '#1B3139'
        },
        'plot_bgcolor': '#F9F7F4',
        'paper_bgcolor': '#F9F7F4',
        'title': {
            'font': {
                'size': 24,
                'color': '#1B3139'
            }
        },
        'colorway': ['#98102A', '#FFAB00', '#00A972', '#2272B4', '#303F47'],
        'xaxis': {
            'gridcolor': '#E0E0E0',
            'linecolor': '#E0E0E0',
            'title': {'font': {'size': 14}}
        },
        'yaxis': {
            'gridcolor': '#E0E0E0',
            'linecolor': '#E0E0E0',
            'title': {'font': {'size': 14}}
        },
        'legend': {
            'bgcolor': '#F9F7F4',
            'bordercolor': '#E0E0E0'
        }
    }
}

# Set the default theme
pio.templates["databricks"] = go.layout.Template(databricks_theme)
pio.templates.default = "databricks"

print("Databricks-inspired Plotly theme has been set as default.")


Databricks-inspired Plotly theme has been set as default.


In [18]:
# Import necessary libraries
import plotly.graph_objects as go

# Select a single time series (e.g., the sixth one)
# Define Databricks colors
databricks_colors = ['#98102A', '#FFAB00', '#00A972', '#2272B4', '#303F47']
unique_id = Y_train_df['unique_id'].unique()[40]

# Filter data for the selected time series
train_data = Y_train_df[Y_train_df['unique_id'] == unique_id]
test_data = Y_test_df[Y_test_df['unique_id'] == unique_id]
pred_data = Y_hat_df[Y_hat_df['unique_id'] == unique_id]

# Create the plot
fig = go.Figure(layout=go.Layout(template="databricks"))

# Add actual data trace
fig.add_trace(go.Scatter(x=train_data['ds'], y=train_data['y'],
                         mode='lines', name='Train'))
fig.add_trace(go.Scatter(x=test_data['ds'], y=test_data['y'],
                         mode='lines', name='Test'))

# Add NBEATS prediction trace
fig.add_trace(go.Scatter(x=pred_data['ds'], y=pred_data['NBEATS'],
                         mode='lines', name='NBEATS Prediction'))

# Update layout
fig.update_layout(title=f'Time Series Forecast for {unique_id}',
                  xaxis_title='Timestamp',
                  yaxis_title='Value',
                  legend=dict(font=dict(size=10)),
                  height=500,
                  width=800)

# Show the plot
fig.show()